<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter04/04_rayleigh_flow_ai_solver.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Interactive Rayleigh-Flow Solver with an ML Inverse Surrogate

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 4:** One-Dimensional Flow with Heat Transfer (Rayleigh Flow)<br>
**Concepts:** Rayleigh-star property ratios · heat-addition choking · subsonic and supersonic branches · ML inverse surrogate versus an analytical baseline

## Learning objectives

- Compute Rayleigh-flow property ratios relative to the sonic state.
- Determine whether a prescribed heat input chokes the flow.
- Compare a learned inverse map with a branch-aware bisection solution.
- Interpret temperature, pressure, density, velocity, and stagnation-pressure changes.

## How to use this notebook

Run the single code cell, wait for the three small neural networks to train, then move the four sliders.

> **Model scope.** The neural networks are surrogates trained on exact Rayleigh relations; they do not add new governing physics and must be judged against the analytical result shown in every valid case.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Layout, VBox, HBox, HTML
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from IPython.display import display, clear_output

# =============================================================================
# 0) Settings (Font 22 as requested)
# =============================================================================
np.random.seed(42)
plt.rcParams.update({
    'font.size': 22,
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 20,
    'figure.figsize': (24, 14),
    'lines.linewidth': 4,
    'lines.markersize': 14
})

# =============================================================================
# 1) Physics Helper Functions
# =============================================================================
def get_gamma_R():
    return 1.4, 287.0

def get_T0_ratio(M, gamma=1.4):
    """Calculates T0 / T0* for a given Mach number."""
    M = np.array(M, dtype=float)
    return (2*(gamma+1)*M**2 / (1+gamma*M**2)**2) * (1 + (gamma-1)/2 * M**2)

def rayleigh_ratios(M):
    """Returns 5 columns: [T/T*, p/p*, rho/rho*, u/u*, p0/p0*]"""
    gamma, _ = get_gamma_R()
    M = np.array(M, dtype=float)
    g = gamma

    p_ps = (g + 1.0) / (1.0 + g * M**2)
    T_Ts = ((g + 1.0)**2 * M**2) / (1.0 + g * M**2)**2
    rho_rhos = (1.0 + g * M**2) / ((g + 1.0) * M**2)
    u_us = (g + 1.0) * M**2 / (1.0 + g * M**2)

    term1 = (g + 1.0) / (1.0 + g * M**2)
    term2 = ((2.0 / (g + 1.0)) * (1.0 + (g - 1.0) * M**2 / 2.0)) ** (g / (g - 1.0))
    p0_p0s = term1 * term2

    return np.column_stack([T_Ts, p_ps, rho_rhos, u_us, p0_p0s])

def solve_analytical_M2(M1, T1, q):
    """Exact numerical solution using bisection."""
    gamma, R = get_gamma_R()
    cp = gamma * R / (gamma - 1)

    T01 = T1 * (1 + 0.5*(gamma-1)*M1**2)
    T02 = T01 + q / cp

    ratio_1 = get_T0_ratio(M1, gamma)
    T0_star = T01 / ratio_1

    if T02 > T0_star * 1.000001:
        return None, True, T0_star # Choked

    target = T02 / T0_star

    if M1 < 1.0:
        low, high = M1, 0.9999999
    else:
        low, high = 1.0000001, M1

    for _ in range(100):
        mid = (low + high) / 2
        val = get_T0_ratio(mid, gamma)
        diff = val - target

        if M1 < 1.0:
            if diff < 0: low = mid
            else: high = mid
        else:
            if diff < 0: high = mid
            else: low = mid

    return (low+high)/2, False, T0_star

# =============================================================================
# 2) TRAINING (High Precision)
# =============================================================================
print("🤖 Training High-Precision Neural Networks...")
gamma, _ = get_gamma_R()

# --- A. Subsonic Inverter (Inverse Physics) ---
print("   1. Training Subsonic Inverter (Physics-Guided)...")
m_sub = np.concatenate([
    np.linspace(0.01, 0.8, 5000),
    np.linspace(0.8, 0.9999, 10000)
])
y_sub = get_T0_ratio(m_sub, gamma)

# INVERTER: Input is Ratio (y), Output is Mach (m)
scaler_X_sub = StandardScaler().fit(y_sub.reshape(-1, 1))
scaler_Y_sub = StandardScaler().fit(m_sub.reshape(-1, 1))

nn_sub = MLPRegressor(hidden_layer_sizes=(128, 128), activation='relu', max_iter=2000, tol=1e-7, random_state=42)
nn_sub.fit(scaler_X_sub.transform(y_sub.reshape(-1, 1)), scaler_Y_sub.transform(m_sub.reshape(-1, 1)).ravel())

# --- B. Supersonic Inverter (Inverse Physics) ---
print("   2. Training Supersonic Inverter (Physics-Guided)...")
m_sup = np.concatenate([
    np.linspace(1.0001, 1.5, 10000),
    np.linspace(1.5, 8.0, 5000)
])
y_sup = get_T0_ratio(m_sup, gamma)

# INVERTER: Input is Ratio (y), Output is Mach (m)
scaler_X_sup = StandardScaler().fit(y_sup.reshape(-1, 1))
scaler_Y_sup = StandardScaler().fit(m_sup.reshape(-1, 1))

nn_sup = MLPRegressor(hidden_layer_sizes=(128, 128), activation='relu', max_iter=2000, tol=1e-7, random_state=42)
nn_sup.fit(scaler_X_sup.transform(y_sup.reshape(-1, 1)), scaler_Y_sup.transform(m_sup.reshape(-1, 1)).ravel())

# --- C. Properties Network (Forward but High Precision) ---
print("   3. Training Properties Network (High Precision)...")
m_prop = np.linspace(0.01, 8.0, 8000)
ratios_prop = rayleigh_ratios(m_prop)
y_prop_log = np.log(ratios_prop)

scaler_X_prop = StandardScaler().fit(m_prop.reshape(-1, 1))
scaler_Y_prop = StandardScaler().fit(y_prop_log)

nn_props = MLPRegressor(hidden_layer_sizes=(128, 128), activation='tanh', max_iter=3000, tol=1e-7, random_state=42)
nn_props.fit(scaler_X_prop.transform(m_prop.reshape(-1, 1)), scaler_Y_prop.transform(y_prop_log))

print("✅ Training Complete.")

# =============================================================================
# 3) Simulation Logic
# =============================================================================
def run_simulation(M1, T1, p1_kPa, q_kJ):
    gamma, R = get_gamma_R()
    cp = gamma * R / (gamma - 1)

    p1 = p1_kPa * 1000.0
    q = q_kJ * 1000.0

    # 1. Calculate Target Physics Ratio
    T01 = T1 * (1 + 0.5*(gamma-1)*M1**2)
    T02 = T01 + q / cp

    ratio_1 = get_T0_ratio(M1, gamma)
    T0_star = T01 / ratio_1

    target_ratio = T02 / T0_star

    # Choking Check
    choked = False
    if target_ratio > 1.000001:
        choked = True
        M2_ai = 1.0
    else:
        # 2. AI Inversion (Solving for M2)
        X_in = np.array([[target_ratio]])

        if M1 < 1.0: # Subsonic Branch
            pred_s = nn_sub.predict(scaler_X_sub.transform(X_in))
            M2_ai = scaler_Y_sub.inverse_transform(pred_s.reshape(-1, 1))[0][0]
        else: # Supersonic Branch
            pred_s = nn_sup.predict(scaler_X_sup.transform(X_in))
            M2_ai = scaler_Y_sup.inverse_transform(pred_s.reshape(-1, 1))[0][0]

    # 3. AI Properties Calculation
    X_prop_in = np.array([[M2_ai]])
    log_props = nn_props.predict(scaler_X_prop.transform(X_prop_in))
    props_ai_ratios = np.exp(scaler_Y_prop.inverse_transform(log_props))[0]

    # Dimensionalize
    r1_full = rayleigh_ratios([M1])[0]
    T_s = T1 / r1_full[0]
    p_s = p1 / r1_full[1]
    rho_s = (p1/(R*T1)) / r1_full[2]
    u_s = (M1 * np.sqrt(gamma*R*T1)) / r1_full[3]
    p01_val = p1 * (1 + 0.5*(gamma-1)*M1**2)**(gamma/(gamma-1))
    p0_s = p01_val / r1_full[4]

    vals_ai = [
        props_ai_ratios[0] * T_s, props_ai_ratios[1] * p_s,
        props_ai_ratios[2] * rho_s, props_ai_ratios[3] * u_s, props_ai_ratios[4] * p0_s
    ]

    # 4. Exact Check
    M2_ex, is_choked_ex, _ = solve_analytical_M2(M1, T1, q)
    vals_ex = [None]*5
    if not is_choked_ex and M2_ex is not None:
        r2 = rayleigh_ratios([M2_ex])[0]
        vals_ex = [
            r2[0]*T_s, r2[1]*p_s, r2[2]*rho_s, r2[3]*u_s, r2[4]*p0_s
        ]

    return {
        "M1": M1, "inlet": [T1, p1, p1/(R*T1), M1*np.sqrt(gamma*R*T1), p01_val],
        "M2_ai": M2_ai, "vals_ai": vals_ai,
        "M2_ex": M2_ex, "vals_ex": vals_ex, "choked": choked,
        "star": [T_s, p_s, rho_s, u_s, p0_s],
        "q_max": cp*(T0_star - T01)
    }

def update_plot(M1, T1, p1_kPa, q_kJ):
    res = run_simulation(M1, T1, p1_kPa, q_kJ)

    fig, axs = plt.subplots(2, 3, figsize=(24, 14))
    axs = axs.ravel()

    M_grid = np.linspace(0.01, 6.0, 500)
    ratios_grid = rayleigh_ratios(M_grid)

    titles = [
        (r"Temperature ($T/T^*$)", r"$T/T^*$"),
        (r"Pressure ($p/p^*$)", r"$p/p^*$"),
        (r"Density ($\rho/\rho^*$)", r"$\rho/\rho^*$"),
        (r"Velocity ($u/u^*$)", r"$u/u^*$"),
        (r"Stagnation Pressure ($p_0/p_0^*$)", r"$p_0/p_0^*$")
    ]

    r1 = rayleigh_ratios([M1])[0]

    for i, (ttl, ylab) in enumerate(titles):
        ax = axs[i]
        # Background
        ax.plot(M_grid, ratios_grid[:, i], 'k-', alpha=0.4, label='Analytical')
        # Inlet
        ax.plot(M1, r1[i], 'go', markersize=14, label='Inlet (1)', zorder=5)

        # AI Result
        ai_ratio = res["vals_ai"][i] / res["star"][i]
        ax.plot(res["M2_ai"], ai_ratio, 'r*', markersize=26, zorder=10, label='AI Predicted')

        # Exact Result
        if not res["choked"] and res["M2_ex"] is not None:
            ex_ratio = res["vals_ex"][i] / res["star"][i]
            ax.plot(res["M2_ex"], ex_ratio, 'o', color='blue', markerfacecolor='none',
                    markeredgewidth=3, markersize=18, zorder=15, label='Exact')

        ax.set_title(ttl, fontweight='bold')
        ax.set_xlabel("Mach Number")
        ax.set_ylabel(ylab)
        ax.grid(True, linestyle='--', alpha=0.5)
        # SHOW LEGEND ON ALL PLOTS
        ax.legend(loc='best', framealpha=0.9)

    axs[5].axis('off')

    # HTML Output
    if res["choked"]:
        status = "<span style='color:red;'>⚠️ CHOKED FLOW</span>"
        err_str = "N/A"
        ex_strs = ["--"]*5
        m2_ex_s = "--"
    else:
        status = "<span style='color:green;'>✅ Valid Solution</span>"
        err_val = 100 * abs(res['M2_ai'] - res['M2_ex']) / (res['M2_ex'] + 1e-9)
        err_str = f"{err_val:.4f}%"
        m2_ex_s = f"{res['M2_ex']:.4f}"
        ex = res["vals_ex"]
        ex_strs = [f"{ex[0]:.1f}", f"{ex[1]/1000:.1f}", f"{ex[2]:.3f}", f"{ex[3]:.1f}", f"{ex[4]/1000:.1f}"]

    ai = res["vals_ai"]

    html = f"""
    <div style="background:#f4f4f4; padding:20px; border-radius:10px; border:2px solid #ccc;">
        <h2>🧠 Physics-Guided AI Solver (Inverter Logic)</h2>
        <p style="font-size:22px;">Status: {status} | <b>Max Heat allowed: {res['q_max']/1000:.1f} kJ/kg</b></p>
        <p style="font-size:20px;">Mach Error: {err_str}</p>
        <table style="width:100%; font-size:22px; text-align:center; border-collapse:collapse;">
            <tr style="background:#ddd; border-bottom:2px solid #999;">
                <th style="padding:10px;">Property</th>
                <th style="padding:10px; color:green;">Inlet (1)</th>
                <th style="padding:10px; color:red;">Outlet (AI)</th>
                <th style="padding:10px; color:blue;">Outlet (Exact)</th>
            </tr>
            <tr>
                <td><b>Mach</b></td>
                <td>{M1:.2f}</td>
                <td><b>{res['M2_ai']:.4f}</b></td>
                <td>{m2_ex_s}</td>
            </tr>
            <tr>
                <td><b>T (K)</b></td>
                <td>{T1:.1f}</td>
                <td>{ai[0]:.1f}</td>
                <td>{ex_strs[0]}</td>
            </tr>
            <tr>
                <td><b>P (kPa)</b></td>
                <td>{p1_kPa:.1f}</td>
                <td>{ai[1]/1000:.1f}</td>
                <td>{ex_strs[1]}</td>
            </tr>
            <tr>
                <td><b>Rho (kg/m³)</b></td>
                <td>{res['inlet'][2]:.3f}</td>
                <td>{ai[2]:.3f}</td>
                <td>{ex_strs[2]}</td>
            </tr>
            <tr>
                <td><b>Vel (m/s)</b></td>
                <td>{res['inlet'][3]:.1f}</td>
                <td>{ai[3]:.1f}</td>
                <td>{ex_strs[3]}</td>
            </tr>
             <tr>
                <td><b>P0 (kPa)</b></td>
                <td>{res['inlet'][4]/1000:.1f}</td>
                <td>{ai[4]/1000:.1f}</td>
                <td>{ex_strs[4]}</td>
            </tr>
        </table>
    </div>
    """
    display(HTML(html))
    plt.tight_layout()
    plt.show()

# =============================================================================
# 4) Controls
# =============================================================================
w_M1 = FloatSlider(value=0.5, min=0.1, max=4.0, step=0.05, description='Inlet Mach', layout=Layout(width='90%'))
w_T1 = FloatSlider(value=300, min=200, max=800, step=10, description='T1 (K)', layout=Layout(width='90%'))
w_p1 = FloatSlider(value=101, min=50, max=500, step=1, description='p1 (kPa)', layout=Layout(width='90%'))
w_q = FloatSlider(value=100, min=0, max=2000, step=10, description='Heat q (kJ/kg)', layout=Layout(width='90%'))

ui = VBox([
    HTML("<h3>🎛️ Control Panel (High Precision Inverter)</h3>"),
    w_M1, w_T1, w_p1, w_q
], layout=Layout(border='2px solid #ccc', padding='15px', margin='10px 0px'))

out = widgets.interactive_output(update_plot, {'M1': w_M1, 'T1': w_T1, 'p1_kPa': w_p1, 'q_kJ': w_q})

display(ui, out)


## Interpretation and verification checklist

- For heat addition, both subsonic and supersonic branches should move toward Mach 1.
- The reported maximum heat input marks the sonic limit for the selected inlet state.
- Treat the exact bisection result—not the smoother ML curve—as the reference.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
